# Evaluating the Solar Detector

Characterises the 5-stage computer-vision detector on the 68 scanned homes in
`eval_data.csv`, and is deliberately **honest about what can and cannot be
measured**.

## TL;DR
- The detector is essentially a **colour detector** — the colour stage separates
  its own *has-solar* vs *no-solar* verdicts far more than any other stage.
- It is **never highly confident** (max combined confidence ≈ 55%).
- **No precision/recall confusion matrix is reported, on purpose.** Ground-truth
  labelling of rooftop solar from zoom-20 tiles is unreliable even for a careful
  human — while building this I mislabelled two *known-solar* homes as "no solar"
  because the panels aren't resolvable at that scale. A confusion matrix built on
  untrustworthy labels would overstate rigour, which is the very thing this
  evaluation exists to avoid.

> Validating this detector properly needs a human-verified labelled set (or
> higher-res imagery). That same data gap is why the heuristic is hard to beat
> and why a trained CNN is the real next step.

In [ ]:
import csv, statistics as st
from collections import Counter

rows = list(csv.DictReader(open("eval_data.csv")))
def num(v):
    try: return float(v)
    except (TypeError, ValueError): return None
print(len(rows), "scanned homes")

In [ ]:
# Detector verdict distribution
for k, v in Counter(r["solar_status"] for r in rows).most_common():
    print(f"{k:10} {v}")

## Which stages actually drive the decision?
Mean score for each stage, split by the detector's own verdict. A big gap = that
stage discriminates; a small gap = it's mostly noise for this decision.

In [ ]:
stages = ["color", "grid", "rect", "blob", "ref"]
groups = {"has_solar": [], "no_solar": []}
for r in rows:
    if r["solar_status"] in groups:
        groups[r["solar_status"]].append(r)

print(f"{'stage':7}{'has_solar':>11}{'no_solar':>11}{'gap':>8}")
for s in stages:
    hs = [num(r[s]) for r in groups["has_solar"] if num(r[s]) is not None]
    ns = [num(r[s]) for r in groups["no_solar"] if num(r[s]) is not None]
    print(f"{s:7}{st.mean(hs):>11.3f}{st.mean(ns):>11.3f}{st.mean(hs)-st.mean(ns):>8.3f}")

**Result (computed on the committed data):**

| stage | has_solar mean | no_solar mean | gap |
|---|---|---|---|
| **color** | **0.573** | **0.143** | **+0.430** |
| grid | 0.382 | 0.322 | +0.060 |
| rect | 0.238 | 0.168 | +0.070 |
| blob | 0.208 | 0.089 | +0.119 |
| ref | 0.624 | 0.520 | +0.104 |

`color` does almost all the work (+0.43). `grid` barely separates the classes
(+0.06) — parallel lines are everywhere in suburbia. This is the quantitative
version of the detector's blind spot: dark / low-contrast panels that don't
register on colour slip straight through.

In [ ]:
# Confidence distribution + operating-point (threshold) sensitivity
conf = [num(r["solar_confidence"]) for r in rows if num(r["solar_confidence"]) is not None]
print(f"confidence%  min={min(conf):.1f}  max={max(conf):.1f}  mean={st.mean(conf):.1f}  median={st.median(conf):.1f}\n")
print("homes flagged at each combined-confidence threshold:")
for t in [20, 25, 30, 36, 40, 50, 60]:
    print(f"  >= {t:>2}% : {sum(c >= t for c in conf):>2} homes")

**Result:** confidence tops out at **54.8%** (mean 24%) — the detector is never
strongly sure. The production threshold is **36%**. The sweep is the
threshold-tuning view: drop to 20% and 37/68 homes get flagged; raise to 60% and
none do. Without trustworthy labels this can't become a precision/recall curve,
but it shows the operating-point trade-off directly.

## Limitations
1. **No trustworthy ground truth** — hand-labelling solar from zoom-20 tiles is
   unreliable (demonstrated by mislabelling known-solar homes), so no confusion
   matrix / precision / recall is claimed.
2. **Mildly self-referential** — the Stage-5 reference library includes some of
   these homes, so `ref` is a little optimistic.
3. **Small, LA/SF-skewed sample (n = 68).**

## To extend
`eval_data.csv` has every per-stage score. With `pip install pandas matplotlib`
you can box-plot each stage by verdict and overlay the 36% threshold — kept as a
stdlib text summary here so the notebook runs with no extra dependencies.